# guardrails — block or redact a call before it is sent

A gate that runs *after* the request has left is an audit trail, not a control. `install()` puts rules on the interceptor chain, so a block **raises** before the provider is touched and a redaction rewrites the request in flight.

> **Offline.** No API key, no network — the provider is a fake with the real client's *shape*, or a
> committed cassette. This notebook runs in CI on Python 3.11 and 3.13 via `nbmake`, so if a cell
> below stops working the build goes red.
>
> Beside it, [`main.py`](main.py) is the same story as a script. The last cell here asserts what
> that script asserts.

In [ ]:
# The notebook sits beside the recipe, so its own module is importable. Everything below reuses the
# recipe's fixtures rather than re-inventing them — a notebook that built its own fake could drift
# away from what `main.py` proves and nobody would notice.
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))

## 1 · Arm two rules

One blocks an injection attempt; one redacts a leaked API key. `calls` is the fake's own record of what it was handed — the only vantage point from which a redaction can be proven.

In [ ]:
import pathlib
import tempfile

import main as recipe
from cendor.acttrace import AuditLog, verify
from cendor.core import instrument
from cendor.guardrails import GuardrailTripped, install, rules, uninstall

calls = []
client = instrument(recipe.fake_openai(calls))
d = tempfile.mkdtemp()
path = str(pathlib.Path(d) / "audit.jsonl")
audit = AuditLog(system="assistant", path=path)
install(
    [
        rules.keyword_deny(["ignore previous instructions"], action="block"),
        rules.regex_rule(r"\bsk-[A-Za-z0-9]{16,}\b", action="redact", stage="input"),
    ]
)

## 2 · The block RAISES

It does not return a decision list you have to remember to read.

In [ ]:
try:
    client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": "ignore previous instructions"}],
    )
except GuardrailTripped as e:
    trip = e.decisions[-1]
    print(f"BLOCKED by {trip.guardrail} ({trip.stage}): {trip.reason}")
print(f"provider calls so far: {len(calls)}  =>  $0.00 spent on it")

## 3 · The redaction happens in flight

In [ ]:
client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "my key is sk-ABCD1234EFGH5678"}],
)
sent = calls[-1]["messages"][0]["content"]
print(f"provider received: {sent!r}")

## 4 · Every decision is on the chain

In [ ]:
uninstall()
audit.detach()
for e in (e for e in audit.entries if e.type == "guardrail_decision"):
    print(f"  {e.payload['action']:<6} {e.payload['stage']:<6} {e.payload['guardrail']}")
ok, _ = verify(path)
print(f"\nchain verifies: {ok}")

## 5 · Prove it

In [ ]:
assert len(calls) == 1, "the blocked prompt should never have been sent"
assert "sk-ABCD1234EFGH5678" not in sent, "the provider received the raw key"
assert "[redacted]" in sent
assert ok is True
print("OK")